# NLP-Based Chatbot

A simple NLP chatbot built using the BANKING77 dataset, TF-IDF vectorization, and Logistic Regression to classify banking-related user queries into intents and generate appropriate responses.



In [ ]:
pip install "datasets<4" #since newer versions do not support older script-based datasets like Banking77

In [48]:
import random
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [49]:
#Step 1 : Loading Banking77 dataset from Hugging Face Datasets Hub from PolyAI respository
print("Loading Banking77 dataset")
dataset=load_dataset("PolyAI/banking77")

#Splitting dataset - the dataset already has the train, test examples in it
X_train=dataset["train"]["text"]
X_test=dataset["test"]["text"]
y_train=dataset["train"]["label"]
Y_test=dataset["test"]["label"]

label_names = dataset["train"].features["label"].names

Loading Banking77 dataset


In [50]:
#Step 2: Convert text into TF-IDF features
print("Vectorizing Text")
vectorizer=TfidfVectorizer(
    lowercase=True,
    stop_words="english",   #to remove common words like 'the', 'is', 'and'
    ngram_range=(1,2))      #uses both single words & 2-word combinations
X_train_vec=vectorizer.fit_transform(X_train) #fit_transform learns vocabulary from training data and transforms text into numeric vectors
X_test_vec=vectorizer.transform(X_test) #transform does not learn but transforms text to vectors

Vectorizing Text


In [51]:
#Step 3: Train Logistic Regression model
print("Training model - Logistic Regression")
model=LogisticRegression(max_iter=2000) #2000 iterations to allow the model to update it weights while learning since TF-IDF can be large and sparse
model.fit(X_train_vec,Y_train)

Training model - Logistic Regression


LogisticRegression(max_iter=2000)

In [52]:
#Step 4: Evaluate Model
Y_pred=model.predict(X_test_vec)
accuracy=accuracy_score(Y_test,Y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.8198


In [53]:
#Step 5: Response dictionary for the banking intent question
intent_responses={
    "card_arrival": [
        "Your card may still be in transit. Please check the delivery status.",
        "It looks like your card has not arrived yet. Please verify the shipping timeline."
    ],
    "cash_withdrawal": [
        "You can withdraw cash using your card at supported ATMs.",
        "This seems related to cash withdrawal. Please check ATM access and limits."
    ],
    "cash_withdrawal_charge": [
        "There may be a withdrawal fee depending on the ATM or account type.",
        "This looks related to withdrawal charges. Please review the fee details."
    ],
    "beneficiary_not_allowed": [
        "The beneficiary could not be added. Please verify the recipient details.",
        "It seems the beneficiary is not allowed. Double-check the account information."
    ],
    "beneficiary_not_defined": [
        "The beneficiary details may be missing. Please add the recipient information first.",
        "This seems to be a beneficiary setup issue. Please define the beneficiary before retrying."
    ],
    "balance_not_updated_after_cheque_or_bank_transfer": [
        "Your balance may take some time to update after a transfer.",
        "It looks like the transfer is still being processed. Please wait a little longer."
    ],
    "transfer_not_received_by_recipient": [
        "The recipient may not have received the transfer yet. Please allow some processing time.",
        "It seems the transfer is delayed. Please verify the transfer status and recipient details."
    ],
    "beneficiary_not_verified": [
        "The beneficiary may not be verified yet. Please complete the verification process.",
        "It looks like beneficiary verification is pending."
    ],
    "cash_withdrawal_card": [
        "Please ensure your card is active and supported for cash withdrawal.",
        "This seems related to cash withdrawal using your card. Please check your card status."
    ],
    "pending_transfer": [
        "Your transfer is still pending. Please wait for processing to complete.",
        "It looks like the transfer has not been completed yet."
    ],
    "declined_transfer": [
        "Your transfer appears to have been declined. Please verify your account balance and details.",
        "This seems like a declined transfer. Please review the entered information and try again."
    ],
    "exchange_rate": [
        "Exchange rates may vary over time. Please check the latest rate in your banking app.",
        "This seems related to exchange rates. Please verify the current rate before proceeding."
    ],
    "beneficiary_not_visible": [
        "The beneficiary may not be visible yet. Please refresh and verify the saved details.",
        "It seems the beneficiary is not showing up. Please check whether it was added successfully."
    ],
    "cash_withdrawal_limit": [
        "Your cash withdrawal may be limited based on your account or card settings.",
        "This looks related to withdrawal limits. Please check your daily or monthly limit."
    ]
}


In [54]:
#Step 6: Default response if predicted intent is not in the dictionary
default_responses=[
    "I understood your banking request. Please check the relevant section in your banking app.",
    "I identified your banking issue. Please review the related service or account details.",
    "I recognized your request, but I need more banking-specific support information to respond better."
]

#checks and responds if the predicted intent from the user input exists in the response dictionary
def get_bot_response(intent_name):
  if intent_name in intent_responses:
    return random.choice(intent_responses[intent_name])
  return random.choice(default_responses)+f"(Predicted intent: {intent_name})"


In [55]:
#This is for the general greetings and responses for them
def detect_general_intent(user_input):
    text = user_input.lower().strip()

    if any(word in text for word in ["hi", "hello", "hey", "good morning", "good evening"]):
        return "greeting"
    if "how are you" in text:
        return "status"
    if any(word in text for word in ["thanks", "thank you"]):
        return "thanks"
    if any(word in text for word in ["bye", "goodbye", "exit", "quit"]):
        return "goodbye"
    if "help" in text:
        return "help"

    return "banking"

In [57]:
#Step 7: Chatbot
print("Banking Chatbot. Type 'quit to exit")
#print("Chatbot: Hello! I am your banking assistant. How can I help you today?")
while True:
  user_input=input("You: ").strip()
  general_intent = detect_general_intent(user_input)

  if general_intent == "greeting":
      print("Chatbot: Hello! I am your banking assistant. How can I help you today?\n")
      continue

  if general_intent == "status":
      print("Chatbot: I am doing well. I am here to help with your banking queries.\n")
      continue

  if general_intent == "thanks":
      print("Chatbot: You're welcome!\n")
      continue

  if general_intent == "help":
      print("Chatbot: I can help with card issues, transfers, cash withdrawal, exchange rates, and beneficiary-related queries.\n")
      continue

  if general_intent == "goodbye":
      print("Chatbot: Goodbye!")
      break

  if not user_input:
      print("Bot: Please type something.")
      continue

  #Transform input
  user_vec=vectorizer.transform([user_input])

  #Predict intent
  pred_label=model.predict(user_vec)[0]
  pred_intent=label_names[pred_label]

  #Get chatbot response
  bot_reply=get_bot_response(pred_intent)

  print(f"Chatbot: {bot_reply}")
  print(f"Predicted intent: {pred_intent}\n")

Banking Chatbot. Type 'quit to exit
You: Hi
Chatbot: Hello! I am your banking assistant. How can I help you today?

You: how are you?
Chatbot: I am doing well. I am here to help with your banking queries.

You: My card has not arrived
Chatbot: It looks like your card has not arrived yet. Please verify the shipping timeline.
Predicted intent: card_arrival

You: Why is my transfer pending?
Chatbot: It looks like the transfer has not been completed yet.
Predicted intent: pending_transfer

You: I cannot add a beneficiary
Chatbot: The beneficiary could not be added. Please verify the recipient details.
Predicted intent: beneficiary_not_allowed

You: Is there any cash withdrawal charge?
Chatbot: This looks related to withdrawal charges. Please review the fee details.
Predicted intent: cash_withdrawal_charge

You: what is the exchange rate?
Chatbot: Exchange rates may vary over time. Please check the latest rate in your banking app.
Predicted intent: exchange_rate

You: Thanks
Chatbot: You're